In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
payor_table = dbutils.widgets.get("payor_table")
billto_table = dbutils.widgets.get("billto_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW demographics_src AS
SELECT
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(MedRecNbr AS STRING) AS MedRecNbr,
  CAST(PhysicianName AS STRING) AS PhysicianName,
  NULL AS PhysicianNPI,
  CAST(PhysicianID AS STRING) AS PhysicianID,
  NULL AS AdmitPhysicianName,
  NULL AS AdmitPhysicianNPI,
  NULL AS AdmitPhysicianID,
  CAST(ReferPhysicianName AS STRING) AS ReferPhysicianName,
  NULL AS ReferPhysicianNPI,
  CAST(ReferPhysicianID AS STRING) AS ReferPhysicianID,
  NULL AS PrimarySurgeonName,
  NULL AS PrimarySurgeonNPI,
  NULL AS PrimarySurgeonID,
  CAST(AdmitDate AS DATE) AS AdmitDate,
  CAST(DischargeDate AS DATE) AS DischargeDate,
  NULL AS Registrar,
  CAST(ARStatus AS STRING) AS ARStatus,
  CAST(PatType AS STRING) AS PatType,
  CAST(Location AS STRING) AS Location,
  NULL AS DRGCode,
  NULL AS DRGDesc,
  NULL AS DRGWeight,
  NULL AS DRGVersion,
  NULL AS AdmitFC,
  NULL AS AdmitFinClass,
  CAST(CurrentFC AS STRING) AS CurrentFC,
  NULL AS CurrentFinClass,
  CAST(TotalCharges AS DOUBLE) AS TotalCharges,
  CAST(TotalPymt AS DOUBLE) AS TotalPymt,
  NULL AS TotalInsPymt,
  NULL AS TotalPatientPymt,
  CAST(TotalAdj AS DOUBLE) AS TotalAdj,
  NULL AS TotalInsAdj,
  NULL AS TotalPatientAdj,
  CAST(AcctBalance AS DOUBLE) AS AcctBalance,
  NULL AS InsBalance,
  NULL AS PatBalance,
  CAST(ExpNetRev AS DOUBLE) AS ExpNetRev,
  CAST(PatFName AS STRING) AS PatFName,
  CAST(PatMName AS STRING) AS PatMName,
  CAST(PatLName AS STRING) AS PatLName,
  NULL AS PatSuffix,
  CAST(PatDOB AS DATE) AS PatDOB,
  CAST(PatSSN AS STRING) AS PatSSN,
  CAST(PatGender AS STRING) AS PatGender,
  CAST(PatAddr1 AS STRING) AS PatAddr1,
  CAST(PatAddr2 AS STRING) AS PatAddr2,
  CAST(PatCity AS STRING) AS PatCity,
  CAST(PatState AS STRING) AS PatState,
  CAST(PatZip AS STRING) AS PatZip,
  NULL AS PatCountry,
  NULL AS PatProvince,
  CAST(PatHomePhone AS STRING) AS PatHomePhone,
  NULL AS PatWorkPhone,
  NULL AS PatCellPhone,
  NULL AS PatEmployer,
  NULL AS GuarRelationship,
  NULL AS Guarantor,
  NULL AS GuarFName,
  NULL AS GuarMName,
  NULL AS GuarLName,
  NULL AS GuarSuffix,
  NULL AS GuarDOB,
  NULL AS GuarSSN,
  NULL AS GuarGender,
  NULL AS GuarAddr1,
  NULL AS GuarAddr2,
  NULL AS GuarCity,
  NULL AS GuarState,
  NULL AS GuarZip,
  NULL AS GuarCountry,
  NULL AS GuarProvince,
  NULL AS GuarHomePhone,
  NULL AS GuarWorkPhone,
  NULL AS GuarCellPhone,
  NULL AS GuarEmployer,
  CAST(LastBillDate AS INT) AS LastBillDate,
  NULL AS LastBillSubmitDate,
  NULL AS LastBillType,
  NULL AS LastBillMediaType,
  NULL AS Agency,
  NULL AS AgencyAssignDate,
  NULL AS AgencyReturnDate,
  NULL AS AgencyReturnReason,
  NULL AS BadDebtDate,
  NULL AS BadDebtAmt,
  NULL AS BadDebtBal,
  CAST(ActiveCOB AS STRING) AS ActiveCOB,
  NULL AS PatientFacilityID,
  NULL AS PatientFacilityName,
  NULL AS PatientFacilityNPI,
  NULL AS PatientFacilityType,
  NULL AS CBSA,
  NULL AS LocationCode,
  NULL AS PatientStatusatBill,
  NULL AS PatientStatusatCodeBill,
  NULL AS LastClaimComment,
  CAST(StatementFromDate AS INT) AS StatementFromDate,
  CAST(StatementThroughDate AS INT) AS StatementThroughDate,
  CAST(DateOfServiceFromDate AS INT) AS DateOfServiceFromDate,
  CAST(DateOfServiceThroughDate AS INT) AS DateOfServiceThroughDate,
  NULL AS MedicalDirectorName,
  NULL AS MedicalDirectorNPI,
  NULL AS MedicalDirectorID,
  NULL AS BenefitPeriodID,
  NULL AS BenefitPeriodStartDate,
  NULL AS BenefitPeriodEndDate,
  NULL AS BenefitPeriodStatus,
  NULL AS EpisodeID,
  NULL AS StartofEpisode,
  NULL AS EndofEpisode,
  NULL AS StartofPeriod,
  NULL AS EndofPeriod,
  NULL AS EpisodeStatus,
  CAST(ClaimFromDate AS INT) AS ClaimFromDate,
  CAST(ClaimThroughDate AS INT) AS ClaimThroughDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH 
  ob_base AS (
    SELECT *
    FROM {source_table}
    WHERE account_balance != 0
    AND date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  demographics_cte AS (
    SELECT
      to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
      ofc.OfficeNumber AS FacilityCode,
      CASE
        WHEN UPPER(obd.invoice_number) = 'ADV'
          THEN CONCAT('ADV - ', clt.SourceSystemId)
        ELSE obd.invoice_number
      END AS AcctNbr,
      clt.SourceSystemId AS MedRecNbr,
      CONCAT(
        TRIM(REPLACE(clt.PrimaryPhysicianFirstName, '"', ' ')),
        ' ',
        TRIM(REPLACE(clt.PrimaryPhysicianLastName, '"', ' '))
      ) AS PhysicianName,
      clt.PrimaryPhysicianCode AS PhysicianID,
      CONCAT(
        TRIM(REPLACE(clt.PrimaryPhysicianFirstName, '"', ' ')),
        ' ',
        TRIM(REPLACE(clt.PrimaryPhysicianLastName, '"', ' '))
      ) AS ReferPhysicianName,
      clt.PrimaryPhysicianCode AS ReferPhysicianID,
      clt.AdmissionDate AS AdmitDate,
      try_to_date(clt.DischargeDate, 'yyyy-MM-dd') AS DischargeDate,
      'Open' AS ARStatus,
      ofc.Practice AS PatType,
      ofc.OfficeState AS Location,
      obd.payor_type_code AS CurrentFC,
      --py.PayorTypeDescription AS CurrentFinClass,
      NULL AS CurrentFinClass,
      CAST(obd.orig_bill AS DOUBLE) AS TotalCharges,
      obd.total_payments AS TotalPymt,
      obd.total_adjustments AS TotalAdj,
      obd.account_balance AS AcctBalance,
      obd.net_revenue AS ExpNetRev,
      TRIM(REPLACE(clt.FirstName, '"', ' ')) AS PatFName,
      clt.MiddleName AS PatMName,
      TRIM(REPLACE(clt.LastName, '"', ' ')) AS PatLName,
      DATE_FORMAT(clt.ConformedBirthDate, 'yyyy-MM-dd') AS PatDOB,
      clt.ConformedSocialSecurityNumber AS PatSSN,
      CASE
        WHEN clt.ConformedGender IN ('Male','Female')
          THEN SUBSTRING(clt.ConformedGender, 1, 1)
        ELSE NULL
      END AS PatGender,
      REPLACE(clt.Address1, '|', '') AS PatAddr1,
      TRIM(REPLACE(clt.Address2, '"', ' ')) AS PatAddr2,
      clt.City AS PatCity,
      clt.State AS PatState,
      clt.Zipcode AS PatZip,
      clt.Phone1 AS PatHomePhone,
      obd.invoice_date_key AS LastBillDate,
      '1' AS ActiveCOB,
      obd.first_visit_date_key AS StatementFromDate,
      obd.last_visit_date_key AS StatementThroughDate,
      obd.first_visit_date_key AS DateOfServiceFromDate,
      obd.last_visit_date_key AS DateOfServiceThroughDate,
      obd.first_visit_date_key AS ClaimFromDate,
      obd.last_visit_date_key AS ClaimThroughDate,
      0 AS SourceSystemKey

    FROM ob_base obd
    LEFT JOIN {office_table} ofc
        ON ofc.OfficeKey = obd.office_key
    LEFT JOIN prd_bronze_raw.mart_bkp.client clt
        ON clt.ClientKey = obd.client_key
    LEFT JOIN {payor_table} py
        ON py.PayorKey = obd.payor_key
    LEFT JOIN {billto_table} bt
        ON py.PayorID = bt.BillToId
  ),
  demographics_clean AS (
    SELECT *,
      ROW_NUMBER() OVER (
          PARTITION BY
            ReportingDate,
            FacilityCode,
            AcctNbr,
            MedRecNbr,
            PhysicianName,
            AdmitDate,
            PatType,
            CurrentFC,
            CurrentFinClass,
            TotalCharges,
            TotalPymt,
            TotalAdj,
            AcctBalance,
            ExpNetRev,
            PatFName,
            PatLName,
            PatGender,
            PatAddr1,
            PatHomePhone
          ORDER BY FacilityCode, AcctNbr, MedRecNbr
      ) AS rn
      FROM demographics_cte
  )
  SELECT *
  FROM demographics_clean
  WHERE rn=1
) AS src
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING demographics_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.MedRecNbr = src.MedRecNbr,
    tgt.PhysicianName = src.PhysicianName,
    tgt.PhysicianNPI = src.PhysicianNPI,
    tgt.PhysicianID = src.PhysicianID,
    tgt.AdmitPhysicianName = src.AdmitPhysicianName,
    tgt.AdmitPhysicianNPI = src.AdmitPhysicianNPI,
    tgt.AdmitPhysicianID = src.AdmitPhysicianID,
    tgt.ReferPhysicianName = src.ReferPhysicianName,
    tgt.ReferPhysicianNPI = src.ReferPhysicianNPI,
    tgt.ReferPhysicianID = src.ReferPhysicianID,
    tgt.PrimarySurgeonName = src.PrimarySurgeonName,
    tgt.PrimarySurgeonNPI = src.PrimarySurgeonNPI,
    tgt.PrimarySurgeonID = src.PrimarySurgeonID,
    tgt.AdmitDate = src.AdmitDate,
    tgt.DischargeDate = src.DischargeDate,
    tgt.Registrar = src.Registrar,
    tgt.ARStatus = src.ARStatus,
    tgt.PatType = src.PatType,
    tgt.Location = src.Location,
    tgt.DRGCode = src.DRGCode,
    tgt.DRGDesc = src.DRGDesc,
    tgt.DRGWeight = src.DRGWeight,
    tgt.DRGVersion = src.DRGVersion,
    tgt.AdmitFC = src.AdmitFC,
    tgt.AdmitFinClass = src.AdmitFinClass,
    tgt.CurrentFC = src.CurrentFC,
    tgt.CurrentFinClass = src.CurrentFinClass,
    tgt.TotalCharges = src.TotalCharges,
    tgt.TotalPymt = src.TotalPymt,
    tgt.TotalInsPymt = src.TotalInsPymt,
    tgt.TotalPatientPymt = src.TotalPatientPymt,
    tgt.TotalAdj = src.TotalAdj,
    tgt.TotalInsAdj = src.TotalInsAdj,
    tgt.TotalPatientAdj = src.TotalPatientAdj,
    tgt.AcctBalance = src.AcctBalance,
    tgt.InsBalance = src.InsBalance,
    tgt.PatBalance = src.PatBalance,
    tgt.ExpNetRev = src.ExpNetRev,
    tgt.PatFName = src.PatFName,
    tgt.PatMName = src.PatMName,
    tgt.PatLName = src.PatLName,
    tgt.PatSuffix = src.PatSuffix,
    tgt.PatDOB = src.PatDOB,
    tgt.PatSSN = src.PatSSN,
    tgt.PatGender = src.PatGender,
    tgt.PatAddr1 = src.PatAddr1,
    tgt.PatAddr2 = src.PatAddr2,
    tgt.PatCity = src.PatCity,
    tgt.PatState = src.PatState,
    tgt.PatZip = src.PatZip,
    tgt.PatCountry = src.PatCountry,
    tgt.PatProvince = src.PatProvince,
    tgt.PatHomePhone = src.PatHomePhone,
    tgt.PatWorkPhone = src.PatWorkPhone,
    tgt.PatCellPhone = src.PatCellPhone,
    tgt.PatEmployer = src.PatEmployer,
    tgt.GuarRelationship = src.GuarRelationship,
    tgt.Guarantor = src.Guarantor,
    tgt.GuarFName = src.GuarFName,
    tgt.GuarMName = src.GuarMName,
    tgt.GuarLName = src.GuarLName,
    tgt.GuarSuffix = src.GuarSuffix,
    tgt.GuarDOB = src.GuarDOB,
    tgt.GuarSSN = src.GuarSSN,
    tgt.GuarGender = src.GuarGender,
    tgt.GuarAddr1 = src.GuarAddr1,
    tgt.GuarAddr2 = src.GuarAddr2,
    tgt.GuarCity = src.GuarCity,
    tgt.GuarState = src.GuarState,
    tgt.GuarZip = src.GuarZip,
    tgt.GuarCountry = src.GuarCountry,
    tgt.GuarProvince = src.GuarProvince,
    tgt.GuarHomePhone = src.GuarHomePhone,
    tgt.GuarWorkPhone = src.GuarWorkPhone,
    tgt.GuarCellPhone = src.GuarCellPhone,
    tgt.GuarEmployer = src.GuarEmployer,
    tgt.LastBillDate = src.LastBillDate,
    tgt.LastBillSubmitDate = src.LastBillSubmitDate,
    tgt.LastBillType = src.LastBillType,
    tgt.LastBillMediaType = src.LastBillMediaType,
    tgt.Agency = src.Agency,
    tgt.AgencyAssignDate = src.AgencyAssignDate,
    tgt.AgencyReturnDate = src.AgencyReturnDate,
    tgt.AgencyReturnReason = src.AgencyReturnReason,
    tgt.BadDebtDate = src.BadDebtDate,
    tgt.BadDebtAmt = src.BadDebtAmt,
    tgt.BadDebtBal = src.BadDebtBal,
    tgt.ActiveCOB = src.ActiveCOB,
    tgt.PatientFacilityID = src.PatientFacilityID,
    tgt.PatientFacilityName = src.PatientFacilityName,
    tgt.PatientFacilityNPI = src.PatientFacilityNPI,
    tgt.PatientFacilityType = src.PatientFacilityType,
    tgt.CBSA = src.CBSA,
    tgt.LocationCode = src.LocationCode,
    tgt.PatientStatusatBill = src.PatientStatusatBill,
    tgt.PatientStatusatCodeBill = src.PatientStatusatCodeBill,
    tgt.LastClaimComment = src.LastClaimComment,
    tgt.StatementFromDate = src.StatementFromDate,
    tgt.StatementThroughDate = src.StatementThroughDate,
    tgt.DateOfServiceFromDate = src.DateOfServiceFromDate,
    tgt.DateOfServiceThroughDate = src.DateOfServiceThroughDate,
    tgt.MedicalDirectorName = src.MedicalDirectorName,
    tgt.MedicalDirectorNPI = src.MedicalDirectorNPI,
    tgt.MedicalDirectorID = src.MedicalDirectorID,
    tgt.BenefitPeriodID = src.BenefitPeriodID,
    tgt.BenefitPeriodStartDate = src.BenefitPeriodStartDate,
    tgt.BenefitPeriodEndDate = src.BenefitPeriodEndDate,
    tgt.BenefitPeriodStatus = src.BenefitPeriodStatus,
    tgt.EpisodeID = src.EpisodeID,
    tgt.StartofEpisode = src.StartofEpisode,
    tgt.EndofEpisode = src.EndofEpisode,
    tgt.StartofPeriod = src.StartofPeriod,
    tgt.EndofPeriod = src.EndofPeriod,
    tgt.EpisodeStatus = src.EpisodeStatus,
    tgt.ClaimFromDate = src.ClaimFromDate,
    tgt.ClaimThroughDate = src.ClaimThroughDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    MedRecNbr,
    PhysicianName,
    PhysicianNPI,
    PhysicianID,
    AdmitPhysicianName,
    AdmitPhysicianNPI,
    AdmitPhysicianID,
    ReferPhysicianName,
    ReferPhysicianNPI,
    ReferPhysicianID,
    PrimarySurgeonName,
    PrimarySurgeonNPI,
    PrimarySurgeonID,
    AdmitDate,
    DischargeDate,
    Registrar,
    ARStatus,
    PatType,
    Location,
    DRGCode,
    DRGDesc,
    DRGWeight,
    DRGVersion,
    AdmitFC,
    AdmitFinClass,
    CurrentFC,
    CurrentFinClass,
    TotalCharges,
    TotalPymt,
    TotalInsPymt,
    TotalPatientPymt,
    TotalAdj,
    TotalInsAdj,
    TotalPatientAdj,
    AcctBalance,
    InsBalance,
    PatBalance,
    ExpNetRev,
    PatFName,
    PatMName,
    PatLName,
    PatSuffix,
    PatDOB,
    PatSSN,
    PatGender,
    PatAddr1,
    PatAddr2,
    PatCity,
    PatState,
    PatZip,
    PatCountry,
    PatProvince,
    PatHomePhone,
    PatWorkPhone,
    PatCellPhone,
    PatEmployer,
    GuarRelationship,
    Guarantor,
    GuarFName,
    GuarMName,
    GuarLName,
    GuarSuffix,
    GuarDOB,
    GuarSSN,
    GuarGender,
    GuarAddr1,
    GuarAddr2,
    GuarCity,
    GuarState,
    GuarZip,
    GuarCountry,
    GuarProvince,
    GuarHomePhone,
    GuarWorkPhone,
    GuarCellPhone,
    GuarEmployer,
    LastBillDate,
    LastBillSubmitDate,
    LastBillType,
    LastBillMediaType,
    Agency,
    AgencyAssignDate,
    AgencyReturnDate,
    AgencyReturnReason,
    BadDebtDate,
    BadDebtAmt,
    BadDebtBal,
    ActiveCOB,
    PatientFacilityID,
    PatientFacilityName,
    PatientFacilityNPI,
    PatientFacilityType,
    CBSA,
    LocationCode,
    PatientStatusatBill,
    PatientStatusatCodeBill,
    LastClaimComment,
    StatementFromDate,
    StatementThroughDate,
    DateOfServiceFromDate,
    DateOfServiceThroughDate,
    MedicalDirectorName,
    MedicalDirectorNPI,
    MedicalDirectorID,
    BenefitPeriodID,
    BenefitPeriodStartDate,
    BenefitPeriodEndDate,
    BenefitPeriodStatus,
    EpisodeID,
    StartofEpisode,
    EndofEpisode,
    StartofPeriod,
    EndofPeriod,
    EpisodeStatus,
    ClaimFromDate,
    ClaimThroughDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.MedRecNbr,
    src.PhysicianName,
    src.PhysicianNPI,
    src.PhysicianID,
    src.AdmitPhysicianName,
    src.AdmitPhysicianNPI,
    src.AdmitPhysicianID,
    src.ReferPhysicianName,
    src.ReferPhysicianNPI,
    src.ReferPhysicianID,
    src.PrimarySurgeonName,
    src.PrimarySurgeonNPI,
    src.PrimarySurgeonID,
    src.AdmitDate,
    src.DischargeDate,
    src.Registrar,
    src.ARStatus,
    src.PatType,
    src.Location,
    src.DRGCode,
    src.DRGDesc,
    src.DRGWeight,
    src.DRGVersion,
    src.AdmitFC,
    src.AdmitFinClass,
    src.CurrentFC,
    src.CurrentFinClass,
    src.TotalCharges,
    src.TotalPymt,
    src.TotalInsPymt,
    src.TotalPatientPymt,
    src.TotalAdj,
    src.TotalInsAdj,
    src.TotalPatientAdj,
    src.AcctBalance,
    src.InsBalance,
    src.PatBalance,
    src.ExpNetRev,
    src.PatFName,
    src.PatMName,
    src.PatLName,
    src.PatSuffix,
    src.PatDOB,
    src.PatSSN,
    src.PatGender,
    src.PatAddr1,
    src.PatAddr2,
    src.PatCity,
    src.PatState,
    src.PatZip,
    src.PatCountry,
    src.PatProvince,
    src.PatHomePhone,
    src.PatWorkPhone,
    src.PatCellPhone,
    src.PatEmployer,
    src.GuarRelationship,
    src.Guarantor,
    src.GuarFName,
    src.GuarMName,
    src.GuarLName,
    src.GuarSuffix,
    src.GuarDOB,
    src.GuarSSN,
    src.GuarGender,
    src.GuarAddr1,
    src.GuarAddr2,
    src.GuarCity,
    src.GuarState,
    src.GuarZip,
    src.GuarCountry,
    src.GuarProvince,
    src.GuarHomePhone,
    src.GuarWorkPhone,
    src.GuarCellPhone,
    src.GuarEmployer,
    src.LastBillDate,
    src.LastBillSubmitDate,
    src.LastBillType,
    src.LastBillMediaType,
    src.Agency,
    src.AgencyAssignDate,
    src.AgencyReturnDate,
    src.AgencyReturnReason,
    src.BadDebtDate,
    src.BadDebtAmt,
    src.BadDebtBal,
    src.ActiveCOB,
    src.PatientFacilityID,
    src.PatientFacilityName,
    src.PatientFacilityNPI,
    src.PatientFacilityType,
    src.CBSA,
    src.LocationCode,
    src.PatientStatusatBill,
    src.PatientStatusatCodeBill,
    src.LastClaimComment,
    src.StatementFromDate,
    src.StatementThroughDate,
    src.DateOfServiceFromDate,
    src.DateOfServiceThroughDate,
    src.MedicalDirectorName,
    src.MedicalDirectorNPI,
    src.MedicalDirectorID,
    src.BenefitPeriodID,
    src.BenefitPeriodStartDate,
    src.BenefitPeriodEndDate,
    src.BenefitPeriodStatus,
    src.EpisodeID,
    src.StartofEpisode,
    src.EndofEpisode,
    src.StartofPeriod,
    src.EndofPeriod,
    src.EpisodeStatus,
    src.ClaimFromDate,
    src.ClaimThroughDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)